# Analyse des images du jeu de données

In [1]:
import pandas as pd

df = pd.read_csv('image_list_clean.csv', dtype={"file":str})

df.head()

,category,type,quality,file,width,height,bands,origin
0,bottle,test,broken_large,000,900,900,3,MVTec
1,bottle,test,broken_large,001,900,900,3,MVTec
2,bottle,test,broken_large,002,900,900,3,MVTec
3,bottle,test,broken_large,003,900,900,3,MVTec
4,bottle,test,broken_large,004,900,900,3,MVTec


# Analyse des tailles et types d'images par catégories

On voit que pour chaque category, on a toujours les mêmes dimensions d'images et le même nombre de bands.
On constate d'ailleurs que toutes les images du jeu de données MVTec sont carrées, mais pas les images "metal_plate" du jeu de données RAD.

In [2]:
import plotly.graph_objects as go
import plotly.io as pio
import numpy as np

try:
    import nbformat  # noqa: F401
except ImportError:
    pio.renderers.default = "browser"

train = df[df["type"] == "train"].groupby("category").agg(count=("file", "count"))
test_good = df[(df["type"] == "test") & (df["quality"] == "good")].groupby("category").agg(count=("file", "count"))
test_bad = df[df["quality"] != "good"].groupby(["category", "quality"]).agg(count=("file", "count"))

bad_pivot = test_bad["count"].unstack("quality").fillna(0)
categories = train.index

x = np.arange(len(categories))
x_good = x-0.2
x_bad  = x+0.2

fig = go.Figure()

# Pile good/train
fig.add_bar(
    x=x_good,
    y=test_good["count"],
    name="test_good",
    marker=dict(color="green")
)

fig.add_bar(
    x=x_good,
    y=train["count"],
    name="train",
    marker=dict(color="lightgreen")
)

# Pile bad
for quality in bad_pivot.columns:
    fig.add_bar(
        x=x_bad,
        y=bad_pivot[quality],
        name=f"bad - {quality}",
    )

fig.update_xaxes(
    tickvals=x,
    ticktext=categories
)

fig.update_layout(
    barmode="stack",
    height=600,
    showlegend=False
)

fig.add_annotation(
    text="train (good)",
    xref="x", yref="y",
    x=4.8, y=350,
    axref="x", ayref="y",
    ax=4.5, ay=370,
    showarrow=True, 
    xanchor="right",
    font=dict(size=14, color="black")
)
fig.add_annotation(
    text="test (good)",
    xref="x", yref="y",
    x=4.8, y=25,
    axref="x", ayref="y",
    ax=4.5, ay=120,
    showarrow=True, 
    xanchor="right",
    font=dict(size=14, color="black")
)
fig.add_annotation(
    text="defects",
    xref="x", yref="y",
    x=5.2, y=68,
    axref="x", ayref="y",
    ax=5.3, ay=120,
    showarrow=True, 
    xanchor="center",
    font=dict(size=14, color="black")
)

fig.update_layout(
    title="Répartition des images par catégorie et par type (train/test) et qualité (good/bad)",
    xaxis_title="Catégorie",
    yaxis_title="Nombre d'images"
)

fig.show()

In [3]:
from plotly.subplots import make_subplots
# Nombre d'erreurs par catégorie et par type d'erreur
# MVTec
counts_mvtec=df[(df["quality"]!="good") & (df["origin"]=='MVTec')].groupby(["category", "quality"]).agg(count=('file','count')).reset_index()
# RAD
counts_rad=df[(df["quality"]!="good") & (df["origin"]=='RAD')].groupby(["category", "quality"]).agg(count=('file','count')).reset_index()

fig = make_subplots(rows=1,cols=2, column_widths=[0.8,0.2])
fig.add_trace(go.Box(
    x=counts_mvtec["category"],
    y=counts_mvtec["count"], name="MVTec"
), row=1, col=1)
fig.add_trace(go.Box(
    x=counts_rad["category"],
    y=counts_rad["count"], name="RAD"
), row=1, col=2)
fig.update_layout(
    title="Distribution du nombre d'images montrant des défauts, par quality",
    xaxis_title="Catégorie",
    yaxis_title="Nombre d'images montrant des défauts, par quality"
)
fig.show()

In [4]:
# Nombre d'erreurs par catégorie et par type d'erreur
counts_cat=df[df["quality"]!="good"].groupby(["category"]).agg(count=('file','count')).reset_index()

fig = make_subplots(rows=2,cols=1)
fig.add_trace(go.Box(
    x=counts_cat["count"],
    orientation="h", name="Toutes les images"
), row=1, col=1)
fig.update_xaxes(title_text="Nombre d'images montrant des défauts, par catégorie", 
                 row=1, col=1)

fig.add_trace(go.Box(
    x=counts_cat.loc[counts_cat["category"]!='metal_plate', "count"],
    orientation="h", name="Images MVTec"
), row=2, col=1)
fig.update_xaxes(title_text="Nombre d'images montrant des défauts, par catégorie", 
                 row=2, col=1)

fig.update_layout(
    title="Distribution du nombre d'images montrant des défauts, par catégorie", 
    showlegend=False
)
fig.show()

In [22]:
# Nombre d'image "train" par catégorie
counts_cat=df[df["type"]=="train"].groupby(["category"]).agg(count=('file','count')).reset_index()

fig=go.Figure()
fig.add_trace(go.Box(
    x=counts_cat["count"], 
    orientation="h", name=''
))
fig.update_layout(
    title="Distribution du nombre d'images d'entraînement par catégorie",
    yaxis_title="Total",
    xaxis_title="Nombre d'images 'train' par catégorie"
)
fig.show()

In [57]:
# Répartition des images RGB et Gris par catégorie 
counts = df.groupby(['category', 'bands']).size().reset_index(name='count')


categories = df['category'].unique()
bands_values = df['bands'].unique()


fig = go.Figure()

for band in bands_values:
    subset = counts[counts['bands'] == band]
    
    subset = subset.set_index('category').reindex(categories, fill_value=0).reset_index()
    
    fig.add_trace(go.Bar(
        x=subset['category'],
        y=subset['count'],
        name="Couleur" if band == 3 else "Niveaux de gris", 
        marker=dict(color='black' if band == 1 else 'lightgreen', 
                    pattern_shape="x" if band==1 else "", 
                    pattern_fillmode="replace", 
                    line_color='black' if band==1 else 'green', 
                    line_width=1)
    ))


fig.update_layout(
    barmode='stack',
    title="Répartition des images RGB et niveaux de gris par catégorie",
    xaxis=dict(
        tickangle=45,
        title="category"
    ),
    yaxis=dict(title="count"),
    legend_title="Mode de couleur", 
    legend=dict(
        yanchor="top",
        y=1,
        xanchor="left",
        x=0.08
    )
)

fig.show()

In [126]:
# Nombre de dimensions par catégorie : width, height, bands
counts = df.groupby(["height", "width", "bands"]).agg(count=('category', 'nunique')).reset_index(names=["height", "width", "bands", "count"])

nb = counts[counts["bands"]==1]
couleur = counts[counts["bands"]==3]

fig = go.Figure()
fig.add_trace(go.Scatter(
        x=couleur['width'],
        y=couleur['height'],
        mode='markers', 
        text=couleur["count"],
        marker=dict(
            size=couleur["count"],  # taille proportionnelle à count
            sizemode='area',  # la surface est proportionnelle à count
            sizeref=2. * counts["count"].max() / (60.**2),
            sizemin=4, 
            color="green"
        ), name="Couleur"
    ))
fig.add_trace(go.Scatter(
        x=nb['width'],
        y=nb['height'],
        mode='markers', 
        text=nb["count"],
        marker=dict(
            size=nb["count"],  # taille proportionnelle à count
            sizemode='area',  # la surface est proportionnelle à count
            sizeref=2. * counts["count"].max() / (60.**2),
            sizemin=4, 
            color="black"
        ), name="Niveau de gris"
    ))
fig.add_annotation(
    yanchor="top",
    y=0.75,
    xanchor="left",
    x=0.05,
    xref="paper",
    yref="paper",
    text="Taille des bulles = nombre de catégories",
    showarrow=False
)

fig.update_layout(
    barmode='stack',
    title="Dimensions des images et couleurs",
    xaxis=dict( title="Largeur de l'image" ),
    yaxis=dict(title="Hauteur de l'image"),
    legend_title="Mode de couleur", 
    legend=dict(
        yanchor="top",
        y=1,
        xanchor="left",
        x=0.05,
        itemsizing='constant'
    )
)
fig.show();


In [16]:
# Nombre d'images par type de défaut
test_bad_MVTec = df[(df["quality"] != "good") & (df["origin"]=="MVTec")].groupby(["category", "quality"]).agg(count=("file", "count"))
test_bad_RAD = df[(df["quality"] != "good") & (df["origin"]=="RAD")].groupby(["category", "quality"]).agg(count=("file", "count"))

fig = make_subplots(rows=2,cols=1)
fig.add_trace(go.Box(
    x=test_bad_MVTec["count"], 
    orientation="h", name=''
), row=1, col=1)
fig.update_xaxes(title_text="Nombre d'images MVTec par type de défaut", 
                 row=1, col=1)

fig.add_trace(go.Box(
    x=test_bad_RAD["count"], 
    orientation="h", name=''
), row=2, col=1)
fig.update_xaxes(title_text="Nombre d'images RAD par type de défaut", 
                 row=2, col=1)

fig.update_layout(
    title="Distribution du nombre d'images avec des défauts, par type de défaut"
)

fig.show()

Analyse du jeu de données RAD

In [1]:
import pandas as pd

df_rad = pd.read_csv('image_list_RAD.csv', dtype={'file':str})
df_rad.head()

,category,type,quality,file,extension,width,height,mode,bands,type_image,mask,mask_bands,mask_mode,mask_type
0,ribbon,test,defect,001097,.png,612,512,RGB,3,C,True,1.0,L,G
1,ribbon,test,defect,000940,.png,612,512,RGB,3,C,True,1.0,L,G
2,ribbon,test,defect,001111,.png,612,512,RGB,3,C,True,1.0,L,G
3,ribbon,test,defect,001099,.png,612,512,RGB,3,C,True,1.0,L,G
4,ribbon,test,defect,001028,.png,612,512,RGB,3,C,True,1.0,L,G


In [ ]:
df_rad['extension'].unique()
# Il n'y a que des png donc on peut supprimer cette colonne 

<StringArray>
['.png']
Length: 1, dtype: str

In [3]:
df_rad = df_rad.drop('extension', axis = 1)

In [ ]:
df_rad[["mask_type", "mask_bands", "mask_mode"]].nunique()

# Les masques sont tous en noir et blanc avec une seule bande,
# On peut tout enlever sauf mask (True/False)

mask_type     1
mask_bands    1
mask_mode     1
dtype: int64

In [6]:
df_rad =  df_rad.drop(["mask_type", "mask_bands", "mask_mode"], axis = 1)

In [ ]:
df_rad.groupby(["mode", "bands", "type_image"]).size()

# Les trois variables décrivent la meme chose donc on peut garder que les bandes 

mode  bands  type_image
RGB   3      C             2375
dtype: int64

In [9]:
df_rad = df_rad.drop(['mode', 'type_image'], axis = 1)

In [ ]:
df_rad['bands'].unique()

# Il n'y que des photos en couleur donc la colonne est redondante et n'apporte pas d'info

array([3])

In [11]:
df_rad = df_rad.drop('bands', axis = 1)

In [ ]:
pd.crosstab(df_rad["quality"], df_rad["mask"])
#il y a 4 images ou il y a un defect mais il n'y pas de mask
# mask n'est donc pas redondant complètement 

mask,False,True
quality,,
defect,4,1224
good,1147,0


In [ ]:
df_rad.groupby("category").agg({"width":("min", "max"), "height": ("min", "max")})
# Il semble que toutes les images font la même taille donc elles sont inutiles
# width: 612, height: 512

width      height     
           min  max    min  max
category                       
bolt       612  612    512  512
ribbon     612  612    512  512
sponge     612  612    512  512
tape       612  612    512  512

In [17]:
df_rad[['width', 'height']].nunique()

width     1
height    1
dtype: int64

In [18]:
df_rad =  df_rad.drop(['width', 'height'], axis = 1)

In [19]:
df_rad.head()

,category,type,quality,file,mask
0,ribbon,test,defect,001097,True
1,ribbon,test,defect,000940,True
2,ribbon,test,defect,001111,True
3,ribbon,test,defect,001099,True
4,ribbon,test,defect,001028,True


In [21]:
df_rad.isna().sum()

category    0
type        0
quality     0
file        0
mask        0
dtype: int64

In [ ]:
import plotly.express as px

#repartition des images par categorie (split train test)
df_plot = df_rad.copy()

df_plot["split"] = df_plot["type"] + " - " + df_plot["quality"]

fig = px.histogram(
    df_plot,
    x="category",
    color="split",
    barmode="stack",
    title="Répartition des images par catégorie"
)

fig.show()

In [ ]:
#Repartition des images par categorie et qualité
# Une meilleur visualisation de cette comparaison que dans le graphique précédent
fig = px.histogram(
    df_rad,
    x="category",
    color="quality",
    barmode="group",
    title="Répartition des images par catégorie et qualité"
)

fig.show()

In [ ]:
#distribution du nombre de defect par cat

df_plot = (
    df_rad[df_rad["quality"] != "good"]
    .groupby("category")
    .size()
    .reset_index(name="count")
)

fig = px.box(
    df_plot,
    y="count",   # 👈 un seul box global
    title="Distribution du nombre d'images en défaut par catégorie"
)

fig.show()

In [ ]:
#nombre d'image d'entrainement par catégorie 

counts_cat = (
    df_rad[df_rad["type"] == "train"]
    .groupby("category")
    .size()
    .reset_index(name="count")
)

fig = px.box(
    counts_cat,
    y="count",
    title="Distribution du nombre d'images d'entraînement par catégorie"
)

fig.show()